In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def autocorrelation_delay(series, max_lag=100):
    """
    Computes autocorrelation and returns the first lag where ACF crosses zero or reaches local minimum.

    Parameters:
    - series: 1D array-like time series
    - max_lag: Maximum number of lags to evaluate

    Returns:
    - optimal delay (int)
    - autocorrelations (array)
    """
    series = np.asarray(series)
    series = (series - np.mean(series)) / np.std(series)  # Normalize

    acf = [1.0]
    for lag in range(1, max_lag + 1):
        corr = np.corrcoef(series[:-lag], series[lag:])[0, 1]
        acf.append(corr)

    acf = np.array(acf)

    # Find first local minimum or first zero crossing
    for i in range(1, len(acf)-1):
        if acf[i] < acf[i-1] and acf[i] < acf[i+1]:
            return i, acf
        if acf[i] < 0:
            return i, acf

    return 1, acf  # Fallback if no clear min or zero-crossing found

# Load data
data = pd.read_csv('/content/Hourly_Resampled/2021.csv')['Chla_Sensor']
data = pd.to_numeric(data, errors='coerce').dropna()

# Compute autocorrelation-based delay
optimal_tau, acf = autocorrelation_delay(data, max_lag=100)

# Plot ACF
plt.figure(figsize=(8, 4))
plt.plot(range(len(acf)), acf, marker='o', linestyle='-', color='blue')
plt.axvline(optimal_tau, color='r', linestyle='--', label=f'Chosen τ = {optimal_tau}')
plt.title("Autocorrelation Function of Chla_Sensor (2021)")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Optimal delay time (τ) from autocorrelation: {optimal_tau}")

In [ ]:
#FNN Estimation 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer
from scipy.spatial import KDTree

def false_nearest_neighbors(time_series, max_dim, tau=9, rtol=10, atol=2):
    N = len(time_series)
    fnn_percentages = []
    for dim in range(1, max_dim + 1):
        num_vectors = N - (dim - 1) * tau
        if num_vectors <= 0:
            break
        embedded_data = np.array([time_series[i : i + dim * tau : tau] for i in range(num_vectors)])
        tree = KDTree(embedded_data)
        false_neighbors = 0
        total_neighbors = 0
        for i in range(len(embedded_data)):
            if i + dim * tau >= N:
                continue
            dist, idx = tree.query(embedded_data[i], k=2)
            if len(idx) < 2 or dist[1] == 0:
                continue
            neighbor_idx = idx[1]
            if neighbor_idx + dim * tau >= N:
                continue
            dist_dim = np.abs(time_series[i + dim * tau] - time_series[neighbor_idx + dim * tau])
            if dist_dim / dist[1] > rtol or dist_dim > atol:
                false_neighbors += 1
            total_neighbors += 1
        fnn_percentage = (false_neighbors / total_neighbors) * 100 if total_neighbors > 0 else 0
        fnn_percentages.append(fnn_percentage)
    return fnn_percentages

# Load & prepare data
data = pd.read_csv('2020.csv')['Chla_Sensor']
imputer = KNNImputer(n_neighbors=2)
data_imputed = imputer.fit_transform(data.values.reshape(-1, 1))
data = pd.Series(data_imputed.flatten())

# Compute FNN
max_dim = 20
fnn_percentages = false_nearest_neighbors(data, max_dim, tau=9)

# Determine “optimal E” — for example: the first dim where FNN is minimal or begins to flatten
# Simple heuristic: choose the dimension with the lowest FNN%
optimal_E = np.nanargmin(fnn_percentages) + 1  # +1 because index 0 maps to dim=1

print("Optimal E:", optimal_E)

# Plot results with annotation
plt.figure(figsize=(8, 4))
x_vals = list(range(1, len(fnn_percentages) + 1))
y_vals = fnn_percentages

plt.plot(x_vals, y_vals, '-o', color='black')
plt.xlabel("Embedding Dimension (E)", fontsize=12)
plt.ylabel("False Nearest Neighbors (%)", fontsize=12)
plt.title("FNN vs Embedding Dimension (2020)", fontsize=14)
plt.ylim(0, 100)
plt.xticks(np.arange(1, max_dim + 1, 1))
plt.grid(True, linestyle='--', alpha=0.6)

# Annotate the optimal E on the plot
y_opt = fnn_percentages[optimal_E - 1]
plt.annotate(f"Optimal E = {optimal_E}",
             xy=(optimal_E, y_opt),
             xytext=(optimal_E + 1, y_opt + 5),
             arrowprops=dict(facecolor='red', arrowstyle='->'),
             fontsize=12,
             ha='left')

plt.tight_layout()
plt.show()

In [ ]:
#Optimal Embedding
import pandas as pd
import pyEDM
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Load and preprocess the data
df = pd.read_csv('Boyd_Chla_daily.csv', parse_dates=['date'])
df.rename(columns={'date': 'Time'}, inplace=True)
df['Time'] = df['Time'].dt.strftime('%Y-%m-%d')
df['Chla_Sensor'] = df['Chla_Sensor'].interpolate(method='linear')
df.dropna(subset=['Chla_Sensor'], inplace=True)

# Define library and prediction ranges
lib_range = "1 1700"
pred_range = "1701 2201"

# Evaluate embedding dimensions from E=1 to E=10
embed_result = pyEDM.EmbedDimension(
    dataFrame=df,
    columns="Chla_Sensor",
    target="Chla_Sensor",
    lib=lib_range,
    pred=pred_range,
    maxE=10,
    Tp=1,
    tau=9,
    showPlot=False
)

# Display the results
print(embed_result)

# Identify optimal embedding dimension (E) based on max rho
optimal_E = embed_result.loc[embed_result['rho'].idxmax(), 'E']
optimal_rho = embed_result['rho'].max()

# Plot rho vs E
plt.figure(figsize=(8, 5))
plt.plot(embed_result['E'], embed_result['rho'], marker='o', color='black')
plt.xlabel('Embedding Dimension (E)', fontsize=12)
plt.ylabel('Prediction Skill (rho)', fontsize=12)
plt.grid(True)

# Annotate optimal E with a red dot
plt.scatter(optimal_E, optimal_rho, color='red', zorder=5)

# Add legend for the red dot
legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='Optimal ε')]
plt.legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from causal_ccm.causal_ccm import ccm
import matplotlib.pyplot as plt

# 1. Load data
df = pd.read_csv("Boyd_Chla_daily.csv", parse_dates=["date"])
df.rename(columns={"date": "Time"}, inplace=True)

# Flatten series
X = df["Temp"].values          # Potential driver
Y = df["Chla_Sensor"].values   # Response

# 2. Choose tau and E
tau = 9     # Delay time from embedding analysis
E = 3       # Embedding dimension from previous optimization
L = len(X)  # Library length (full series)

# 3. Run CCM for X->Y (Temp drives Chl-a)
ccm_XY = ccm(X, Y, tau, E, L)
rho_XY = ccm_XY.causality()  # strength of causal signal

# 4. Run CCM for Y->X (Chl-a drives Temp)
ccm_YX = ccm(Y, X, tau, E, L)
rho_YX = ccm_YX.causality()

print("Temp → Chl-a CCM skill:", rho_XY)
print("Chl-a → Temp CCM skill:", rho_YX)

In [ ]:
import pandas as pd
import numpy as np
from causal_ccm.causal_ccm import ccm
import matplotlib.pyplot as plt

# 1. Load data
df = pd.read_csv("Boyd_daily_all.csv", parse_dates=["date"])
df.rename(columns={"date": "Time"}, inplace=True)

# Extract time series
X = df["Temperature"].values      # Potential driver
Y = df["Chla_Sensor"].values      # Response

# 2. CCM parameters
tau = 9     # time delay from embedding analysis
E = 3       # embedding dimension from previous optimization
L = len(X)  # total length of time series

# Library sizes to test (start from 50 up to full length)
L_sizes = np.arange(50, L, 100)

# Lists to store CCM skill (rho)
rho_X_to_Y = []
rho_Y_to_X = []

for L_size in L_sizes:
    # X → Y (Temperature driving Chl-a)
    ccm_XY = ccm(X, Y, tau, E, L_size)
    rho_X_to_Y.append(ccm_XY.causality()[0])

    # Y → X (Chl-a driving Temperature)
    ccm_YX = ccm(Y, X, tau, E, L_size)
    rho_Y_to_X.append(ccm_YX.causality()[0])

# 3. Plot CCM convergence for both directions
plt.figure(figsize=(8, 6))
plt.plot(L_sizes, rho_X_to_Y, label='Temp → Chl-a', marker='o', color='blue')
plt.plot(L_sizes, rho_Y_to_X, label='Chl-a → Temp', marker='s', color='red')

plt.xlabel("Library Size (L)", size=12)
plt.ylabel("CCM Skill (ρ)", size=12)
plt.title("CCM Convergence (Bidirectional)", size=14)
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
#Getting optimum theta
import pandas as pd
import pyEDM

# Load and preprocess the data
df = pd.read_csv('2015.csv', parse_dates=['date'])
df.rename(columns={'date': 'Time'}, inplace=True)
df['Time'] = df['Time'].dt.strftime('%Y-%m-%d')
df['Chla_Sensor'] = df['Chla_Sensor'].interpolate(method='linear')
df.dropna(subset=['Chla_Sensor'], inplace=True)

# Define library and prediction ranges
lib_range = "1 300"
pred_range = "301 365"

# Run PredictNonlinear
result = pyEDM.PredictNonlinear(
    dataFrame=df,
    columns='Chla_Sensor',
    target='Chla_Sensor',
    lib=lib_range,
    pred=pred_range,
    E=6,
    tau=8,
    showPlot=True
)

# Identify the theta with the maximum prediction skill
max_row = result.loc[result['rho'].idxmax()]
optimal_theta = max_row['Theta']
max_rho = max_row['rho']

print(f"Optimal theta: {optimal_theta}")
print(f"Maximum prediction skill (rho): {max_rho}")

In [ ]:
#Evaluating Non-linearity
import pandas as pd
import matplotlib.pyplot as plt
import pyEDM

# Load and preprocess the data
df = pd.read_csv('2015.csv', parse_dates=['date'])
df.rename(columns={'date': 'Time'}, inplace=True)
df['Time'] = df['Time'].dt.strftime('%Y-%m-%d')
df['Chla_Sensor'] = df['Chla_Sensor'].interpolate(method='linear')
df.dropna(subset=['Chla_Sensor'], inplace=True)

# Define library and prediction ranges
lib_range = "1 300"
pred_range = "301 365"

# Run PredictNonlinear to evaluate prediction skill across theta values
result = pyEDM.PredictNonlinear(
    dataFrame=df,
    columns='Chla_Sensor',
    target='Chla_Sensor',
    lib=lib_range,
    pred=pred_range,
    E=6,
    tau=8,
    showPlot=False  # Set to True if you want pyEDM's default plot
)

# Standardize column names to lowercase to avoid KeyError
result.columns = [col.lower() for col in result.columns]

# Identify the theta with the maximum prediction skill
max_row = result.loc[result['rho'].idxmax()]
optimal_theta = max_row['theta']
max_rho = max_row['rho']

print(f"Optimal theta: {optimal_theta}")
print(f"Maximum prediction skill (rho): {max_rho}")

# Visualize prediction skill across theta values
plt.figure(figsize=(8, 5))
plt.plot(result['theta'], result['rho'], marker='o')
plt.xlabel('Theta')
plt.ylabel('Prediction Skill (rho)')
plt.title('Prediction Skill vs. Theta')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#Simplex
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyEDM

# Load and preprocess the data
df = pd.read_csv('daily_chla.csv', parse_dates=['Date'])
df.rename(columns={'Date': 'Time'}, inplace=True)
df['Time'] = df['Time'].dt.strftime('%Y-%m-%d')
df['Chla_Sensor'] = df['Chla_Sensor'].interpolate(method='linear')
df.dropna(subset=['Chla_Sensor'], inplace=True)

# Set library and prediction ranges
lib_range = "1 300"
pred_range = "301 365"

# Determine optimal embedding dimension (E)
E_range = range(1, 11)
rho_values = []

for E in E_range:
    try:
        result = pyEDM.Simplex(dataFrame=df,
                               lib=lib_range,
                               pred=pred_range,
                               E=E,
                               columns="Chla_Sensor",
                               target="Chla_Sensor",
                               showPlot=False)
        if 'rho' in result.columns:
            rho = result['rho'].values[0]
        else:
            print(f"Simplex failed for E={E}: 'rho' not found in result.")
            rho = np.nan
    except Exception as e:
        print(f"Simplex failed for E={E}: {e}")
        rho = np.nan
    rho_values.append(rho)

# Plot rho vs. E
plt.figure(figsize=(8, 4))
plt.plot(E_range, rho_values, marker='o', linestyle='-')
plt.xlabel('Embedding Dimension (E)')
plt.ylabel('Prediction Skill (rho)')
plt.title('Simplex Projection: rho vs. Embedding Dimension')
plt.grid(True)
plt.show()

# Perform Simplex projection with optimal E
if all(np.isnan(rho_values)):
    print("Simplex failed for all embedding dimensions. Please check your data and parameters.")
else:
    optimal_E = E_range[np.nanargmax(rho_values)]
    print(f"Optimal embedding dimension (E): {optimal_E}")

    simplex_result = pyEDM.Simplex(dataFrame=df,
                                   lib=lib_range,
                                   pred=pred_range,
                                   E=optimal_E,
                                   columns="Chla_Sensor",
                                   target="Chla_Sensor",
                                   showPlot=False)

    # Plot actual vs. predicted values
    plt.figure(figsize=(10, 5))
    plt.plot(simplex_result['Time'], simplex_result['Observations'], label='Observed')
    plt.plot(simplex_result['Time'], simplex_result['Predictions'], label='Predicted')
    plt.xlabel('Time')
    plt.ylabel('Chlorophyll-a Concentration')
    plt.title('Simplex Projection: Observed vs. Predicted')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
#SMAP
import pandas as pd
import matplotlib.pyplot as plt
import pyEDM
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Load and preprocess the data
df = pd.read_csv('2015.csv', parse_dates=['date'])
df.rename(columns={'date': 'Time'}, inplace=True)
df['Time'] = df['Time'].dt.strftime('%Y-%m-%d')
df['Chla_Sensor'] = df['Chla_Sensor'].interpolate(method='linear')
df.dropna(subset=['Chla_Sensor'], inplace=True)

# Define library and prediction ranges
lib_range = "1 300"
pred_range = "301 365"

# Run SMap without plotting
result = pyEDM.SMap(
    dataFrame=df,
    columns='Chla_Sensor',
    target='Chla_Sensor',
    lib=lib_range,
    pred=pred_range,
    E=6,
    Tp=1,
    theta=1.5,
    tau=8,
    showPlot=False
)

# Extract the predictions DataFrame
predictions = result['predictions']

# Reconstruct the 'Time' column if it's missing
if 'Time' not in predictions.columns:
    # Extract the corresponding time values from the original DataFrame
    time_values = df.loc[300:364, 'Time'].values  # Adjust indices for zero-based indexing
    predictions['Time'] = time_values

# Calculate R-squared and RMSE
observed = predictions['Observations'].values
predicted = predictions['Predictions'].values

# Ensure there are no NaN values
mask = ~np.isnan(observed) & ~np.isnan(predicted)
observed_clean = observed[mask]
predicted_clean = predicted[mask]

r_squared = r2_score(observed_clean, predicted_clean)
rmse = mean_squared_error(observed_clean, predicted_clean, squared=False)

print(f"R-squared (R²): {r_squared:.4f}")
print(f"Root Mean Square Error (RMSE): {rmse:.4f}")

# Create a custom plot
plt.figure(figsize=(10, 6))
plt.plot(predictions['Time'], predictions['Observations'], label='Observed', color='blue', linewidth=2)
plt.plot(predictions['Time'], predictions['Predictions'], label='Predicted', color='orange', linestyle='--', linewidth=2)
plt.xlabel('Time')
plt.ylabel('Chlorophyll-a (μg/l)')
plt.title('SMap Projection: Observed vs Predicted')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()